# ADJ：四个新来源、五片段固定确认

γ=1；ADJ、原RANK3R、CDC_M共享观测。仅原几何拒绝时尝试单窗唯一第二名，先限制origin相差1，再看合法路径并以唯一最小分差选择。已有CPU开发结果不等于新源确认。本入口默认直接读取B分支可用源码，用户Run all执行一次新验证。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Drive与一次尝试

已有attempt/启动收据或执行痕迹时只读；失败、中断和未进入项保留，不自动补跑。云端预检未知不启动；quota仅记录。普通启动收据不宣称并发原子锁。


In [ ]:

from pathlib import Path
import json, os, shutil, stat, subprocess, sys, tempfile, urllib.request, zipfile
os.environ.update(HF_HOME='/content/video-wm-cache/huggingface', HF_HUB_CACHE='/content/video-wm-cache/huggingface/hub', TRANSFORMERS_CACHE='/content/video-wm-cache/huggingface/hub', TORCH_HOME='/content/video-wm-cache/torch')
INPUT = Path('/content/drive/MyDrive/Video-WM/B-Rank3-Four-Source-Confirmation-V1/repeat-20261025-v1')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/B-Adjacent-Origin-Confirmation-V1/fixed-four-five-v1')
SOURCE_REF = 'dev/local-joint-state-payload-v1'
SOURCE_DIRECTORY = None
"""Colab repeat storage: actual Drive API reads, separate from method evaluation.

No model, codec, production runtime or network work occurs at import. The
Notebook embeds this readable module for cloud guarding before source setup.
"""
import json
from pathlib import Path
import shutil
import tempfile
import time
import uuid
import zipfile

REPEAT_NAME='fixed-four-five-v1'
DRIVE_COMPONENTS=('Video-WM','B-Adjacent-Origin-Confirmation-V1',REPEAT_NAME)
DEFAULT_OUTPUT='/content/drive/MyDrive/'+'/'.join(DRIVE_COMPONENTS)
CORE_JSON=('result.json','evaluation.json','ledger.json','planned_outputs.json','plan.json','protocol.json',
           'background.json','thresholds.json','physical_counts.json','physical_events.json',
           'plans_complete.json','blind_complete.json','post_truth_access.json','attempt.json','progress.json')
HOSTS=tuple('b-adj-v%02d'%n for n in range(1,5))
MEDIA=tuple(f'media/{h}/{arm}/{scope}/video.mp4' for h in HOSTS for arm in ('OFF','P0','PM')
            for scope in ('full','CROP','DELETE_LEFT_GAP','DELETE_RIGHT_GAP','REPEAT_LEFT_GAP','REPEAT_RIGHT_GAP'))
RGB_ARCHIVES=()
RGB_SIDECARS=()
ARCHIVE_BLOBS=set(RGB_ARCHIVES)|set(RGB_SIDECARS)

ARTIFACT_FOLDERS=('receipts','private','media','observations','families','blind','quality','models','source_media','m4_inputs')
ROOT_ARTIFACTS=set(CORE_JSON)|{'inputs.json','storage.json','candidate_stop.json','source_protocol.json','source_thresholds.json','off_rgb24_archives.json','repeat_start.json','repeat_start_unknown.json'}
FOLDER_MIME='application/vnd.google-apps.folder'


def _json(stream):
    def invalid(value):raise ValueError('non-JSON numeric constant: '+value)
    return json.load(stream,parse_constant=invalid)


def _name(name):
    if not isinstance(name,str) or not name or name in ('.','..') or any(c in name for c in ('/','\\',':','\x00')):
        raise ValueError('unsafe Drive child name')
    return name


def _quoted(value):
    return "'"+str(value).replace('\\','\\\\').replace("'","\\'")+"'"


class DriveClient:
    """Small Drive v3 adapter. Writes only one ordinary startup receipt."""
    def __init__(self,session):self.session=session
    def _metadata(self,method,url,**kwargs):
        response=self.session.request(method,url,timeout=(20,120),**kwargs)
        try:
            response.raise_for_status();return response.json()
        finally:response.close()
    def quota(self):
        return self._metadata('GET','https://www.googleapis.com/drive/v3/about',params={'fields':'storageQuota'}).get('storageQuota',{})
    def children(self,parents):
        query='trashed = false and ('+' or '.join(_quoted(p)+' in parents' for p in parents)+')'
        token=None;answer=[]
        while True:
            params=dict(q=query,spaces='drive',pageSize=1000,fields='nextPageToken,incompleteSearch,files(id,name,mimeType,size,parents)')
            if token:params['pageToken']=token
            value=self._metadata('GET','https://www.googleapis.com/drive/v3/files',params=params)
            if value.get('incompleteSearch'):raise RuntimeError('Drive returned an incomplete directory listing')
            answer.extend(value.get('files',[]));token=value.get('nextPageToken')
            if not token:return answer
    def read_to(self,meta,destination):
        """Read alt=media through EOF. No media decoding and no in-memory video."""
        response=self.session.request('GET','https://www.googleapis.com/drive/v3/files/'+meta['id'],
            params={'alt':'media'},headers={'Accept-Encoding':'identity'},stream=True,timeout=(20,120))
        count=0
        try:
            response.raise_for_status()
            with Path(destination).open('wb') as stream:
                for block in response.iter_content(chunk_size=1024*1024):
                    if block:stream.write(block);count+=len(block)
            if count==0:raise IOError('empty cloud blob')
            expected=meta.get('size')
            if expected is not None and count!=int(expected):raise IOError(f'cloud read length {count} differs from listed bytes {expected}')
            return count
        finally:response.close()
    def create_start(self,parent_id,payload):
        """Exactly one create request. Unknown responses are never retried here."""
        boundary='rank3-'+uuid.uuid4().hex
        metadata=json.dumps(dict(name='repeat_start.json',parents=[parent_id],mimeType='application/json')).encode()
        body=json.dumps(payload,ensure_ascii=False).encode()
        chunks=[b'--'+boundary.encode()+b'\r\nContent-Type: application/json; charset=UTF-8\r\n\r\n'+metadata,
                b'--'+boundary.encode()+b'\r\nContent-Type: application/json\r\n\r\n'+body,
                b'--'+boundary.encode()+b'--\r\n']
        return self._metadata('POST','https://www.googleapis.com/upload/drive/v3/files',
            params={'uploadType':'multipart','fields':'id,name,mimeType,size,parents'},
            headers={'Content-Type':'multipart/related; boundary='+boundary},data=b'\r\n'.join(chunks))


def authenticate_drive():
    from google.colab import auth
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    auth.authenticate_user()
    credentials,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
    return DriveClient(AuthorizedSession(credentials))


def resolve_folder(client,components=DRIVE_COMPONENTS):
    parent='root'
    for name in components:
        matches=[v for v in client.children([parent]) if v.get('name')==name]
        if not matches:return None
        if len(matches)!=1 or matches[0].get('mimeType')!=FOLDER_MIME:
            raise ValueError('ambiguous or non-folder Drive path component: '+name)
        parent=matches[0]['id']
    return parent


def local_artifacts(output):
    output=Path(output);found=[]
    for name in ROOT_ARTIFACTS:
        for suffix in ('','.tmp'):
            p=output/(name+suffix)
            if p.exists() or p.is_symlink():found.append(name+suffix)
    for name in ARTIFACT_FOLDERS:
        folder=output/name
        if folder.is_dir():found.extend(p.relative_to(output).as_posix() for p in folder.rglob('*') if p.is_file() or p.is_symlink())
    return sorted(found)


def cloud_artifacts(client,folder):
    if folder is None:return []
    pending=[(folder,'')];found=[]
    while pending:
        parent,prefix=pending.pop()
        for item in client.children([parent]):
            name=_name(item['name']);relative=prefix+name
            if not prefix:
                if name.removesuffix('.tmp') in ROOT_ARTIFACTS:found.append(relative)
                elif name in ARTIFACT_FOLDERS and item.get('mimeType')==FOLDER_MIME:pending.append((item['id'],name+'/'))
            elif item.get('mimeType')==FOLDER_MIME:pending.append((item['id'],relative+'/'))
            else:found.append(relative)
        if found:return found  # One actual execution trace is enough to prevent a second attempt.
    return found


def preflight(client,output,components=DRIVE_COMPONENTS):
    local=local_artifacts(output);value=dict(status='UNKNOWN',safe_to_start=False,local_artifacts=local,cloud_artifacts=[],folder_id=None,errors=[],quota=None)
    if client is None:
        value['errors'].append('Drive API authentication unavailable');return value
    try:value['quota']=client.quota()
    except Exception as error:value['quota_error']=f'{type(error).__name__}: {error}'
    try:
        value['folder_id']=resolve_folder(client,components)
        value['cloud_artifacts']=cloud_artifacts(client,value['folder_id'])
        value['status']='EXISTING_ATTEMPT' if local or value['cloud_artifacts'] else 'NO_EXECUTION_OBSERVED'
        value['safe_to_start']=value['status']=='NO_EXECUTION_OBSERVED'
    except Exception as error:value['errors'].append(f'{type(error).__name__}: {error}')
    return value


def reserve_start(client,output,components=DRIVE_COMPONENTS,*,attempts=6,pause=5,sleep=time.sleep):
    """Recheck after setup; only a visible, unique mount-created folder is used.

    A receipt is an ordinary once-attempt record, not atomic cross-session exclusion.
    Folder visibility reads may repeat; create and its response are never retried.
    """
    output=Path(output);folder=None
    for index in range(attempts):
        state=preflight(client,output,components)
        if not state['safe_to_start']:return dict(status='NOT_STARTED',preflight=state)
        folder=state['folder_id']
        if folder:break
        if index+1<attempts:sleep(pause)
    if not folder:return dict(status='NOT_STARTED',error='mount-created repeat folder is not yet visible via Drive API; no folder or receipt was created')
    payload=dict(status='STARTING_FIXED_REPEAT',repeat=REPEAT_NAME,output_components=list(components),created_unix=time.time(),
                 purpose='durable one-attempt startup receipt; ADJ four-new-source five-fixture confirmation',atomic_mutex=False)
    try:
        created=client.create_start(folder,payload)
        if not isinstance(created,dict) or not isinstance(created.get('id'),str):raise ValueError('create returned no usable receipt id')
        with tempfile.TemporaryDirectory(prefix='rank3-receipt-') as temporary:
            path=Path(temporary)/'receipt.json';client.read_to(created,path)
            with path.open(encoding='utf-8') as stream:readback=_json(stream)
        if not isinstance(readback,dict) or readback.get('status')!='STARTING_FIXED_REPEAT':raise ValueError('startup receipt is not a readable start record')
        return dict(status='START_RECEIPT_CLOUD_READ_BACK',file_id=created['id'],folder_id=folder,atomic_mutex=False)
    except Exception as error:
        record=dict(status='START_UNKNOWN_NO_WORKER',error=f'{type(error).__name__}: {error}',created_unix=time.time(),retry_create=False)
        # Preserve uncertainty locally if the write may have reached Drive. Never masquerade as runtime attempt.json.
        try:
            with (output/'repeat_start_unknown.json').open('x',encoding='utf-8') as stream:json.dump(record,stream,ensure_ascii=False)
        except Exception as local_error:record['local_record_error']=f'{type(local_error).__name__}: {local_error}'
        return record


def inventory(client,folder):
    """Directory metadata only for supplementary JSON; no per-file content GET."""
    files={};pending={folder:''};visited=set()
    while pending:
        batch=dict(list(pending.items())[:40])
        for ident in batch:pending.pop(ident);visited.add(ident)
        for item in client.children(list(batch)):
            parents=[p for p in item.get('parents',[]) if p in batch]
            if len(parents)!=1:raise ValueError('Drive file has ambiguous listed parent')
            relative=batch[parents[0]]+_name(item['name'])
            if item.get('mimeType')==FOLDER_MIME:
                if item['id'] in visited or item['id'] in pending:raise ValueError('repeated directory id')
                pending[item['id']]=relative+'/'
            else:files.setdefault(relative,[]).append(item)
    return files


def _is_json(name):return name.endswith('.json') or name.endswith('.json.tmp')


def _load_summary(path):
    try:
        with Path(path).open(encoding='utf-8') as stream:value=_json(stream)
        if not isinstance(value,dict):return dict(read_error='root record must be an object')
        return {k:value.get(k) for k in ('status','method_status','contribution_status','stop') if k in value}
    except Exception as error:return dict(read_error=f'{type(error).__name__}: {error}')


def _core_state(name,value):
    """Existing final-state semantics only; no whole-file equality or digest."""
    if not isinstance(value,dict):raise ValueError('core JSON root must be an object')
    state={k:value.get(k) for k in ('status','method_status','contribution_status') if k in value}
    if name=='plan.json':state['units']=sorted(value.get('units',{}))
    elif name=='ledger.json':
        state.update(slots=sorted(value),returned=sum(isinstance(v,dict) and v.get('status')=='RETURNED' and v.get('attempts')==1 for v in value.values()),
            finished=sum(isinstance(v,dict) and v.get('status') in ('RETURNED','FAILED','INTERRUPTED') for v in value.values()),spent=sum(isinstance(v,dict) and v.get('attempts')==1 for v in value.values()))
    elif name=='planned_outputs.json':
        for key in ('tasks','quality'):
            rows=value.get(key,{})
            state[key+'_count']=len(rows);state[key+'_returned']=sum(isinstance(v,dict) and v.get('status')=='RETURNED' for v in rows.values())
            state[key+'_finished']=sum(isinstance(v,dict) and v.get('status') in ('RETURNED','FAILED','INTERRUPTED') for v in rows.values())
        state['reader_slots_valid']=all(set(t.get('readers',{}))=={'ADJ','RANK3R','CDC_M'} and all(len(v.get('slots',[]))==32 for v in t['readers'].values()) for t in value.get('tasks',{}).values())
    elif name=='physical_counts.json':
        state['counts']={k:{n:v.get(n) for n in ('planned','started','returned','failed')} for k,v in value.items() if isinstance(v,dict)}
    elif name in ('plans_complete.json','blind_complete.json'):
        state.update({k:value.get(k) for k in ('task_plans','reader_plans','task_decisions','reader_decisions')})
    elif name=='evaluation.json':
        state['task_count']=len(value.get('tasks',{}));state['quality_count']=len(value.get('quality',{}))
        state['quality_finished']=sum(isinstance(v,dict) and v.get('status') in ('SCORED','RETURNED','FAILED','INTERRUPTED') for v in value.get('quality',{}).values())
    elif name=='physical_events.json':state['progress']=sum((v.get('started') or 0)+(v.get('returned') or 0) for v in value.values() if isinstance(v,dict))
    return state


def _state_score(name,state):
    if name=='result.json':return (state.get('status')=='COMPLETED_FIXED_ATTEMPT',0)
    if name=='evaluation.json':return (state.get('status')=='POST_RECORDED',state.get('task_count',0)+state.get('quality_finished',0))
    if name=='ledger.json':return (False,state.get('finished',0)*2+state.get('spent',0))
    if name=='planned_outputs.json':return (False,state.get('tasks_finished',0)+state.get('quality_finished',0))
    if name=='physical_counts.json':return (False,sum((v.get('started') or 0)+(v.get('returned') or 0)+(v.get('failed') or 0) for v in state.get('counts',{}).values()))
    if name=='physical_events.json':return (False,state.get('progress',0))
    if name=='plan.json':return (False,len(state.get('units',[])))
    return (state.get('status')=='READ_BACK',0)


def _final_state_issues(states,local_states):
    expected=local_states.get('result.json',{}).get('status')=='COMPLETED_FIXED_ATTEMPT' or states.get('result.json',{}).get('status')=='COMPLETED_FIXED_ATTEMPT'
    issues=[]
    if local_states.get('result.json',{}).get('status') in ('COMPLETED_FIXED_ATTEMPT','TECHNICAL_UNRESOLVED','METHOD_NOT_SUPPORTED'):
        if states.get('result.json',{}).get('status')!=local_states['result.json']['status']:issues.append('cloud result differs from the mounted final computation state')
        for name in ('evaluation.json','ledger.json','planned_outputs.json','physical_counts.json','physical_events.json','plan.json','plans_complete.json','blind_complete.json'):
            if _state_score(name,local_states.get(name,{}))>_state_score(name,states.get(name,{})):issues.append('cloud progress is behind mounted final record: '+name)
    if not expected:return issues
    def require(condition,message):
        if not condition:issues.append(message)
    require(states.get('result.json',{}).get('status')=='COMPLETED_FIXED_ATTEMPT','cloud result is not the completed computation record')
    evaluation=states.get('evaluation.json',{})
    require(evaluation.get('status')=='POST_RECORDED' and evaluation.get('task_count')==120 and evaluation.get('quality_count')==108,'cloud evaluation is not final POST_RECORDED with 120 tasks and108 quality records')
    for field in ('method_status','contribution_status'):
        local=local_states.get('evaluation.json',{})
        if local.get('status')=='POST_RECORDED' and field in local:require(local[field]==evaluation.get(field),'cloud/local final '+field+' differ')
    outputs=states.get('planned_outputs.json',{})
    require(outputs.get('tasks_count')==outputs.get('tasks_returned')==120 and outputs.get('quality_count')==outputs.get('quality_returned')==108 and outputs.get('reader_slots_valid') is True,'cloud planned_outputs still contains unfinished or absent fixed rows')
    units=states.get('plan.json',{}).get('units',[]);ledger=states.get('ledger.json',{})
    require(bool(units) and ledger.get('slots')==units and ledger.get('returned')==len(units),'cloud ledger is not returned for the saved fixed plan')
    expected_counts={'native50_trajectory':8,'native_step':400,'DiT_forward':800,'payload_gradient':100,'state_construct':4,'D181':12,'codec_encode':72,'codec_probe':72,'codec_decode':72,'E41':720}
    counts=states.get('physical_counts.json',{}).get('counts',{})
    require(set(counts)==set(expected_counts) and all(counts[k].get('planned')==counts[k].get('started')==counts[k].get('returned')==n and counts[k].get('failed')==0 for k,n in expected_counts.items()),'cloud physical_counts are not the completed fixed budget')
    plans=states.get('plans_complete.json',{});blind=states.get('blind_complete.json',{})
    require(plans.get('status')=='READ_BACK' and plans.get('task_plans')==120 and plans.get('reader_plans')==360,'cloud plans_complete is not final')
    require(blind.get('status')=='READ_BACK' and blind.get('task_decisions')==120 and blind.get('reader_decisions')==360,'cloud blind_complete is not final')
    # attempt STARTED and progress active=post are normal final files in the successful runner.
    return issues


def confirm_and_backup(client,output,backup_directory='/content',components=DRIVE_COMPONENTS,*,progress=None):
    """Read 15 core blobs and 72 media from cloud; back up all visible JSON + 72 MP4.

    Supplementary JSON content comes from the mounted result tree. Cloud-only
    supplementary JSON is listed as missing from backup, never fetched as a new
    per-file cloud verification gate. Missing data never invokes physical work.
    """
    output=Path(output);backup_directory=Path(backup_directory);backup_directory.mkdir(parents=True,exist_ok=True)
    report=dict(status='SAVING',computation=dict(mounted_result=_load_summary(output/'result.json'),mounted_evaluation=_load_summary(output/'evaluation.json')),
        cloud_confirmation=dict(status='UNCONFIRMED',required_core_json=len(CORE_JSON),required_media=len(MEDIA),readable_core_json=0,readable_media=0,
            files={},missing=[],errors=[],supplementary_json_content_verified=False),
        backup=dict(status='INCOMPLETE',scope='all visible JSON and fixed72 MP4; no raw OFF archives' ,files={},missing=[],errors=[]),
        optional_OFF_archives=dict(expected_files=0,expected_bytes=0,readable_RGB24=0,readable_sidecars=0,missing=[],errors=[],method_required=False),
        method_status_modified=False,physical_calls=0)
    cloud=report['cloud_confirmation'];backup=report['backup'];listing={};inventory_ok=False
    local_states={};cloud_states={};valid_local=set()
    if client is not None:
        try:
            folder=resolve_folder(client,components)
            if folder is None:cloud['errors'].append('repeat output folder missing in Drive API')
            else:listing=inventory(client,folder);inventory_ok=True
        except Exception as error:cloud['errors'].append(f'{type(error).__name__}: {error}')
    else:cloud['errors'].append('Drive API unavailable; mounted files cannot establish cloud saving')
    json_local={}
    try:
        if output.is_dir():
            for path in output.rglob('*'):
                if path.is_file() and _is_json(path.name):
                    if path.is_symlink():backup['errors'].append('symlink result file not followed: '+str(path));continue
                    json_local[path.relative_to(output).as_posix()]=path
    except Exception as error:backup['errors'].append(f'local JSON inventory: {type(error).__name__}: {error}')
    for name,path in json_local.items():
        if name in CORE_JSON:
            try:
                with path.open(encoding='utf-8') as stream:local_states[name]=_core_state(name,_json(stream))
                valid_local.add(name)
            except Exception as error:backup['errors'].append(dict(path=name,error='mounted core JSON: '+str(error)))
    expected_json=set(CORE_JSON)|set(RGB_SIDECARS)|set(json_local)|{name for name in listing if _is_json(name)}
    for name,entries in listing.items():
        if len(entries)>1 and (_is_json(name) or name in MEDIA):backup['errors'].append('ambiguous cloud path in backup inventory: '+name)
    cloud['supplementary_json_listed']=len([p for p in listing if _is_json(p) and p not in CORE_JSON])
    required=set(CORE_JSON)|set(MEDIA)|ARCHIVE_BLOBS
    stamp=time.strftime('%Y%m%dT%H%M%S',time.gmtime())+'-'+uuid.uuid4().hex[:8]
    prefix=backup_directory/('b-rank3-'+REPEAT_NAME+'-'+stamp)
    temporary_zip=Path(str(prefix)+'-incomplete.zip');report_path=Path(str(prefix)+'-backup-report.json')
    try:
        with tempfile.TemporaryDirectory(prefix='rank3-cloud-read-',dir=backup_directory) as directory:
            spool=Path(directory);cloud_paths={}
            for index,name in enumerate(sorted(required),1):
                candidates=listing.get(name,[])
                target=report['optional_OFF_archives'] if name in ARCHIVE_BLOBS else cloud
                if len(candidates)!=1:
                    (target['missing'] if not candidates else target['errors']).append(name if not candidates else 'ambiguous cloud path: '+name)
                    continue
                meta=candidates[0];path=spool/str(index)
                try:
                    length=client.read_to(meta,path)
                    if name in ARCHIVE_BLOBS:
                        if name in RGB_ARCHIVES:
                            if length!=88965120:raise IOError('OFF RGB24 archive length is not181*320*512*3')
                            report['optional_OFF_archives']['readable_RGB24']+=1
                        else:
                            with path.open(encoding='utf-8') as stream:sidecar=_json(stream)
                            if not isinstance(sidecar,dict) or sidecar.get('status')!='ARCHIVED' or sidecar.get('shape')!=[181,320,512,3] or sidecar.get('dtype')!='uint8' or sidecar.get('actual_bytes')!=88965120:raise ValueError('OFF RGB24 sidecar does not describe a complete archive')
                            report['optional_OFF_archives']['readable_sidecars']+=1
                    elif name.endswith('.json'):
                        with path.open(encoding='utf-8') as stream:value=_json(stream)
                        cloud_states[name]=_core_state(name,value)
                        if name in ('result.json','evaluation.json'):report['computation']['cloud_'+name.removesuffix('.json')]={k:value.get(k) for k in ('status','method_status','contribution_status','stop') if k in value}
                        cloud['readable_core_json']+=1
                    else:cloud['readable_media']+=1
                    cloud_paths[name]=path;cloud['files'][name]=dict(status='CLOUD_READ_TO_EOF',bytes=length,file_id=meta['id'])
                except Exception as error:target['errors'].append(dict(path=name,error=f'{type(error).__name__}: {error}'))
                if progress and (index%8==0 or index==len(required)):progress(f'Cloud blob reads: {index}/{len(required)}')
            cloud['final_state_issues']=_final_state_issues(cloud_states,local_states)
            if cloud['readable_core_json']==len(CORE_JSON) and cloud['readable_media']==len(MEDIA) and not cloud['missing'] and not cloud['errors'] and not cloud['final_state_issues']:
                cloud['status']='CORE_RESULTS_AND_72_MEDIA_CLOUD_READABLE'
            else:cloud['status']='CORE_OR_MEDIA_CLOUD_UNCONFIRMED'
            with zipfile.ZipFile(temporary_zip,'w',allowZip64=True) as archive:
                for name in sorted(expected_json|set(MEDIA)|set(RGB_ARCHIVES)):
                    local=json_local.get(name) if _is_json(name) else output/name
                    candidate=cloud_paths.get(name);origin='Drive API full read'
                    if _is_json(name) and local is not None and local.is_file():
                        try:
                            if name not in valid_local:
                                with local.open(encoding='utf-8') as stream:_json(stream)
                                valid_local.add(name)
                            if name not in cloud_paths or _state_score(name,local_states.get(name,{}))>=_state_score(name,cloud_states.get(name,{})):
                                candidate=local;origin='mounted readable JSON; selected by saved progress semantics, no whole-file equality'
                        except Exception as error:
                            backup['errors'].append(dict(path=name,error='mounted JSON cannot be parsed: '+str(error)))
                            try:archive.write(local,'unparsed_local/'+name,compress_type=zipfile.ZIP_DEFLATED,compresslevel=3)
                            except Exception as raw_error:backup['errors'].append(dict(path=name,error='cannot preserve malformed original: '+str(raw_error)))
                    if candidate is None:
                        candidate=local;origin='mounted result file; cloud content not confirmed'
                    if candidate is None or not candidate.is_file() or candidate.is_symlink():backup['missing'].append(name);continue
                    try:
                        if name in set(MEDIA)|set(RGB_ARCHIVES) and name not in cloud_paths:
                            length=candidate.stat().st_size
                            if length<=0:raise IOError('empty mounted media/archive')
                            if name in RGB_ARCHIVES and length!=88965120:raise IOError('mounted OFF RGB24 length is incomplete')
                            entries=listing.get(name,[])
                            if len(entries)==1 and entries[0].get('size') is not None and length!=int(entries[0]['size']):raise IOError('mounted MP4 length differs from listed cloud bytes')
                        if _is_json(name) and candidate==local and name not in valid_local:raise ValueError('malformed local JSON is preserved separately, not a complete backup member')
                        archive.write(candidate,'results/'+name,compress_type=zipfile.ZIP_DEFLATED if _is_json(name) else zipfile.ZIP_STORED,compresslevel=3 if _is_json(name) else None)
                        backup['files'][name]=dict(source=origin,bytes=candidate.stat().st_size,cloud_path_readable=name in cloud_paths,backed_up_content_from_cloud=candidate==cloud_paths.get(name))
                    except Exception as error:backup['errors'].append(dict(path=name,error=f'{type(error).__name__}: {error}'))
                backup['json_count']=sum(_is_json(n) for n in backup['files']);backup['media_count']=sum(n in MEDIA for n in backup['files'])
                backup['off_RGB24_count']=sum(n in RGB_ARCHIVES for n in backup['files'])
                backup['cloud_directory_inventory_complete']=inventory_ok
                backup['status']='COMPLETE_BACKUP_OF_LISTED_JSON_AND_72_MEDIA' if inventory_ok and not backup['missing'] and not backup['errors'] and backup['media_count']==72 and backup['off_RGB24_count']==0 else 'INCOMPLETE_BACKUP'
                report['status']='STORAGE_CHECK_FINISHED';report['backup_zip']=str(prefix)+'-complete.zip' if backup['status']=='COMPLETE_BACKUP_OF_LISTED_JSON_AND_72_MEDIA' else str(temporary_zip);report['report_path']=str(report_path)
                archive.writestr('backup_report.json',json.dumps(report,ensure_ascii=False,indent=2),compress_type=zipfile.ZIP_DEFLATED)
        if backup['status']=='COMPLETE_BACKUP_OF_LISTED_JSON_AND_72_MEDIA':
            complete_zip=Path(str(prefix)+'-complete.zip');temporary_zip.rename(complete_zip);report['backup_zip']=str(complete_zip)
    except Exception as error:
        backup['status']='INCOMPLETE_BACKUP';backup['errors'].append(f'backup creation: {type(error).__name__}: {error}')
        report['status']='STORAGE_CHECK_INCOMPLETE';report['backup_zip']=str(temporary_zip) if temporary_zip.is_file() else None
    report['report_path']=str(report_path)
    try:report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    except OSError as error:report['report_write_error']=f'{type(error).__name__}: {error}'
    return report


def storage_summary(report):
    cloud=report['cloud_confirmation'];backup=report['backup']
    return dict(computation=report['computation'],cloud_save_status=cloud['status'],
        cloud_core_json=f"{cloud['readable_core_json']}/{cloud['required_core_json']}",cloud_media=f"{cloud['readable_media']}/{cloud['required_media']}",
        cloud_missing_count=len(cloud['missing']),cloud_error_count=len(cloud['errors']),cloud_final_state_issues=cloud.get('final_state_issues',[]),backup_status=backup['status'],
        backup_json_count=backup.get('json_count',0),backup_media_count=backup.get('media_count',0),off_RGB24_cloud=report.get('optional_OFF_archives'),backup_off_RGB24=backup.get('off_RGB24_count',0),backup_missing_count=len(backup['missing']),
        backup_errors=len(backup['errors']),backup_zip=report.get('backup_zip'),report_path=report.get('report_path'),
        method_status_modified=False,physical_calls=0)

KEYS={'key00':'watermark','key01':'watermark-wrong'}

ARMS=('OFF','P0','PM')

WINDOWS={'L':0,'M':53,'R':106}

FIXTURES={'CROP':(0,None),'DELETE_LEFT_GAP':(8,49),'DELETE_RIGHT_GAP':(8,102),'REPEAT_LEFT_GAP':(-8,49),'REPEAT_RIGHT_GAP':(-8,102)}

READERS=('ADJ','RANK3R','CDC_M')

BUDGET={'native50_trajectory':8,'native_step':400,'DiT_forward':800,'payload_gradient':100,'state_construct':4,'D181':12,'codec_encode':72,'codec_probe':72,'codec_decode':72,'E41':720}

SOURCES=[{'id': 'b-adj-v01', 'seed': 2026102701, 'a': 2, 'category': '湖面、帆船、植被', 'prompt': 'A fixed wide shot of a small red sailboat moving slowly across a calm lake, with green reeds in the foreground and distant wooded hills. The entire sailboat remains visible.', 'negative_prompt': '', 'role': 'eval'}, {'id': 'b-adj-v02', 'seed': 2026102702, 'a': 11, 'category': '日间市场、手部、果蔬', 'prompt': 'A fixed medium shot of a market vendor arranging bright oranges and green apples in wooden crates at an outdoor stall during daylight. The hands and all crates stay in frame.', 'negative_prompt': '', 'role': 'eval'}, {'id': 'b-adj-v03', 'seed': 2026102703, 'a': 17, 'category': '雪景、电车、行进', 'prompt': 'A fixed wide shot of a red tram moving slowly along a snowy residential street while light snow falls. Bare trees and pale buildings remain visible behind the tram.', 'negative_prompt': '', 'role': 'eval'}, {'id': 'b-adj-v04', 'seed': 2026102704, 'a': 6, 'category': '陶艺工坊、缓慢水平平移', 'prompt': 'A slow steady horizontal pan across a pottery workshop bench with clay bowls, ceramic pitchers and wooden sculpting tools in soft daylight. The camera stays level and the lighting remains constant.', 'negative_prompt': '', 'role': 'eval'}]

DENOMINATORS={'sources': 4, 'eval_sources': 4, 'trajectories': 8, 'native_steps': 400, 'DiT_forward': 800, 'local_gradient': 100, 'state_construct': 4, 'D181': 12, 'full_MP4': 12, 'fixture_MP4': 60, 'codec_encode': 72, 'codec_probe': 72, 'codec_decode': 72, 'CPU_edit': 60, 'E41': 720, 'Q': 1440, 'H_families': 360, 'H_score_slots': 49320, 'E41_DCT': 720, 'E41_key_bits': 1440, 'native_key00_Q': 8, 'native_DCT': 8, 'native_key_bits': 16, 'native_P0_PM_bit_pairs': 2816, 'media_P0_PM_bit_pairs': 9600, 'codebooks': 8, 'readers': 3, 'tasks_per_reader': 120, 'reader_plans': 360, 'reader_decisions': 360, 'slots_per_reader': 3840, 'all_reader_slots': 11520, 'positive_tasks': 20, 'positive_slots': 640, 'null_tasks': 100, 'null_slots': 3200, 'wrong_key_PM_tasks': 20, 'wrong_key_PM_slots': 640, 'local_CDC_slots': 11520, 'quality_records': 108, 'quality_frames': 17148, 'quality_adjacent': 17040, 'bundles': 0}

def task_id(h, arm, fixture, key):
    return '/'.join((h, arm, fixture, key))

def tasks():
    return [task_id(h['id'],a,f,k) for h in SOURCES for a in ARMS for f in FIXTURES for k in KEYS]

def quality_specs():
    records={}
    for host in SOURCES:
        h=host['id']
        for pair in ('OFF_to_P0','P0_to_PM'):
            for scope,layers in [('full',('preclip','rgb8','readback'))]+[(f,('rgb8','readback')) for f in FIXTURES]:
                for layer in layers:records[f'{h}/quality/{scope}/{pair}/{layer}']=(h,scope,pair,layer)
        records[f'{h}/quality/full/OFF_to_PM/readback']=(h,'full','OFF_to_PM','readback')
    return records

def plan_units():
    units={'inputs':'input_semantics','background':'background_load','threshold':'old_frozen_tau',
           'plans_complete':'plan_readback','blind_complete':'blind_readback','post':'post_evaluation'}
    for host in SOURCES:
        h=host['id']
        for k in KEYS:units[h+'/book/'+k]='codebook'
        units[h+'/generation_load']='generation_load';units[h+'/generation_release']='generation_release';units[h+'/vae_load']='vae_load';units[h+'/vae_release']='vae_release'
        for a in ('OFF','P0'):units[h+'/generate/'+a]='native50_trajectory'
        for a in ('P0','PM'):
            units[f'{h}/native/{a}/Q']='native_key00_Q'
            units[f'{h}/native/{a}/DCT']='native_DCT'
            for k in KEYS:units[f'{h}/native/{a}/{k}/bits']='native_local_bits'
        units[h+'/construct']='state_construct';units[h+'/native_audit']='native_state_audit'
        for a in ARMS:
            for op,kind in [('D','D181'),('preclip','preclip'),('quantize','quantize'),('encode','codec_encode'),('probe','codec_probe'),('decode','codec_decode')]:
                units[f'{h}/{a}/full/{op}']=kind
            for f in FIXTURES:
                stem=f'{h}/{a}/{f}'
                for op,kind in [('edit','fixture151'),('encode','codec_encode'),('probe','codec_probe'),('decode','codec_decode')]:units[stem+'/'+op]=kind
                for w in WINDOWS:
                    for phase in range(4):
                        view=f'{stem}/{w}/p{phase}';units[view+'/E']='E41';units[view+'/DCT']='E41_DCT'
                        for k in KEYS:
                            units[view+'/'+k+'/Q']='E41_Q';units[view+'/'+k+'/bits']='local_and_Rc_bits'
                    for k in KEYS:units[f'{stem}/{w}/{k}/score']='H137_score'
    for task in tasks():
        units['blind/'+task+'/plan']='blind_plan';units['blind/'+task+'/read']='blind_read'
    for slot in quality_specs():units[slot]='quality_pair'
    return units

def recover_ledger(output):
    """Read-only receipt overlay; also embedded verbatim in the notebook.

    STARTED remains spent and incomplete. Invalid or uncommitted receipts
    remain unresolved, never an inferred successful or unentered call.
    """
    from collections import Counter
    import json
    from pathlib import Path
    output = Path(output)
    ledger={slot:dict(slot=slot,kind=kind,status='NOT_ENTERED',attempts=0) for slot,kind in plan_units().items()}
    errors, receipt_slots = {}, []
    try:
        saved = json.loads((output/'ledger.json').read_text(encoding='utf-8'))
        if not isinstance(saved,dict):raise ValueError('ledger must map slots to records')
        for slot,row in saved.items():
            if slot not in ledger:
                errors['ledger/'+slot]='unknown saved slot';continue
            if not isinstance(row,dict) or row.get('kind')!=ledger[slot]['kind']:
                ledger[slot].update(status='UNRESOLVED_LEDGER',attempts=1 if isinstance(row,dict) and row.get('attempts')==1 else None)
                errors['ledger/'+slot]='invalid saved slot';continue
            status,attempts=row.get('status'),row.get('attempts')
            if status not in ('NOT_ENTERED','STARTED','RETURNED','FAILED','INTERRUPTED') or type(attempts) is not int or attempts!=(0 if status=='NOT_ENTERED' else 1):
                ledger[slot].update(status='UNRESOLVED_LEDGER',attempts=1 if attempts==1 else None);errors['ledger/'+slot]='invalid status/attempt';continue
            ledger[slot]=row
    except (OSError, ValueError) as error:
        errors['ledger.json'] = str(error)
    directory = output/'receipts'
    paths = sorted(directory.rglob('*.json')) + sorted(directory.rglob('*.json.tmp'))
    for path in paths:
        relative = path.relative_to(directory).as_posix()
        pending = relative.endswith('.tmp')
        slot = relative[:-9] if pending else relative[:-5]
        if slot not in ledger:
            errors['receipts/'+relative]='receipt outside fixed plan';continue
        previous = ledger[slot]
        receipt = None
        try:
            receipt = json.loads(path.read_text(encoding='utf-8'))
            if pending:
                raise ValueError('uncommitted receipt; completion is unresolved')
            if not isinstance(receipt, dict) or receipt.get('slot') != slot:
                raise ValueError('receipt slot does not match its address')
            status, attempts = receipt.get('status'), receipt.get('attempts')
            if status not in ('NOT_ENTERED', 'STARTED', 'RETURNED', 'FAILED', 'INTERRUPTED'):
                raise ValueError('invalid receipt status')
            if type(attempts) is not int or attempts != (0 if status == 'NOT_ENTERED' else 1):
                raise ValueError('invalid receipt attempt count')
            if not isinstance(receipt.get('kind'), str) or not receipt['kind']:
                raise ValueError('receipt kind required')
            if previous.get('kind') is not None and previous['kind'] != receipt['kind']:
                raise ValueError('receipt kind disagrees with saved slot')
            if previous.get('attempts') == 1 and attempts == 0:
                raise ValueError('receipt cannot erase a spent call')
            ledger[slot] = receipt
            receipt_slots.append(slot)
        except (OSError, ValueError) as error:
            name = 'receipts/'+relative
            errors[name] = str(error)
            ledger[slot] = dict(previous, slot=slot, status='UNRESOLVED_RECEIPT',
                                attempts=1 if previous.get('attempts') == 1 or isinstance(receipt,dict) and receipt.get('attempts')==1 else None,
                                saved_status=previous.get('status'), recovery_error=name)
    counts = dict(Counter(row.get('status', 'UNRESOLVED_LEDGER') for row in ledger.values()))
    kinds = sorted({row.get('kind', 'UNKNOWN') for row in ledger.values()})
    unit_counts = {kind: dict(Counter(row.get('status', 'UNRESOLVED_LEDGER') for row in ledger.values()
                                     if row.get('kind', 'UNKNOWN') == kind)) for kind in kinds}
    return dict(ledger=ledger, counts=counts, unit_counts=unit_counts,
                receipt_slots=receipt_slots, read_errors=errors, physical_reentry=False,denominators=denominators())

def execution_artifacts(output):
    output=Path(output)
    names=('attempt.json','plan.json','ledger.json','inputs.json','protocol.json','background.json','thresholds.json',
           'physical_counts.json','physical_events.json','progress.json','result.json','evaluation.json','planned_outputs.json','plans_complete.json','blind_complete.json','post_truth_access.json','candidate_stop.json','repeat_start.json','repeat_start_unknown.json','source_protocol.json','source_thresholds.json','off_rgb24_archives.json','storage.json')
    found=[n+s for n in names for s in ('','.tmp') if (output/(n+s)).exists() or (output/(n+s)).is_symlink()]
    for name in ('receipts','private','media','observations','families','blind','quality','source_media','models','m4_inputs'):
        folder=output/name
        if folder.is_dir():found.extend(str(p.relative_to(output)) for p in folder.rglob('*') if p.is_file() or p.is_symlink())
    return sorted(found)

def denominators():return dict(DENOMINATORS)
CLOUD_CLIENT=None
PREPARATION_ERROR=None
SOURCE_ROOT=None
try:
    try:
        import google.auth
        from google.auth.transport.requests import AuthorizedSession
    except ModuleNotFoundError as error:
        if error.name not in ('google','google.auth','google.auth.transport','google.auth.transport.requests','requests'):
            raise
        subprocess.run([sys.executable,'-m','pip','install','google-auth','requests'],check=True)
    CLOUD_CLIENT=authenticate_drive()
except Exception as error:
    PREPARATION_ERROR=f'{type(error).__name__}: {error}'
    print('Drive API认证不可用，本次不启动物理执行:',PREPARATION_ERROR)
PREFLIGHT=preflight(CLOUD_CLIENT,OUTPUT)
CAN_START=PREFLIGHT['safe_to_start']
print('云端/挂载执行痕迹预检:',PREFLIGHT['status'])
print('Drive实时storageQuota（未设容量阈值）:',PREFLIGHT.get('quota'))
print('具体读取问题:',PREFLIGHT.get('errors'),PREFLIGHT.get('quota_error'))
if not CAN_START:print('只读已有结果或报告云端未知状态；不下载源码、不setup、不运行模型。')


## 2. 源码与旧背景

普通B分支直接下载，无需手动上传源码ZIP。复用成功安装、ftfy/torchao定点修复、ffmpeg和FP32 VAE。旧background.json仅作背景输入；新输出独立，不覆盖任何旧阶段。


In [ ]:
if CAN_START:
    try:
        workspace=Path(tempfile.mkdtemp(prefix='b-rank3-source-')).resolve()
        if SOURCE_DIRECTORY is None:
            archive=workspace/'source.zip'
            url='https://codeload.github.com/RICHAAARC/SC-SSTW/zip/refs/heads/'+SOURCE_REF
            urllib.request.urlretrieve(url,archive)
            destination=workspace/'source';destination.mkdir()
            with zipfile.ZipFile(archive) as source:
                for member in source.infolist():
                    normalized=member.filename.replace('\\','/')
                    target=(destination/normalized).resolve()
                    if normalized.startswith('/') or any(':' in p for p in normalized.split('/')) or not target.is_relative_to(destination.resolve()) or stat.S_ISLNK(member.external_attr>>16):
                        raise ValueError('unsafe source archive entry: '+member.filename)
                source.extractall(destination)
            entries=list(destination.glob('*/experiments/wan_state_clock/b_adjacent_origin_confirmation_v1_run.py'))
            if len(entries)!=1:raise FileNotFoundError('RANK3 source entry unavailable: '+url)
            SOURCE_ROOT=entries[0].parents[2]
        else:
            original=Path(SOURCE_DIRECTORY).expanduser().resolve()
            if not (original/'experiments/wan_state_clock/b_adjacent_origin_confirmation_v1_run.py').is_file():
                raise FileNotFoundError('RANK3 entry missing in SOURCE_DIRECTORY')
            SOURCE_ROOT=workspace/'source'
            shutil.copytree(original,SOURCE_ROOT,ignore=shutil.ignore_patterns('.git','__pycache__'))
        OUTPUT.mkdir(parents=True,exist_ok=True)
        print('隔离源码目录:',SOURCE_ROOT)
        print('原始输入:',INPUT,'新输出:',OUTPUT)

    except Exception as error:
        CAN_START=False
        PREPARATION_ERROR=f'{type(error).__name__}: {error}'
        print('源码准备失败，未启动物理执行:',PREPARATION_ERROR)
else:
    print('跳过源码下载：只读结果或云端状态未知。')


## 3. 固定四源与预算

湖面红帆船、日间果蔬摊、雪景红电车、陶艺工坊慢速水平pan；seed2026102701–04，a=2/11/17/6。8轨迹、400步、800DiT、100局部梯度、4构造、D18112、codec各72、E41720、Q1440；native Q8/DCT8/keybits16，观测DCT720/keybits1440，108质量记录。360计划全部回读后读取位，360决定回读后评价。模型与临时tensor在/content。


In [ ]:
if CAN_START:
    try:
        LOG = OUTPUT/'setup.log'
        def command(args, label, check=True):
            print(label, flush=True)
            with LOG.open('a') as stream:
                stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
                completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
            if completed.returncode and check:
                raise RuntimeError(label+' failed; see '+str(LOG))
            return completed.returncode
        def ensure_imports(probe, packages, label):
            import re
            mapping = {p.replace('-', '_'):p for p in packages}
            repaired = set()
            while True:
                offset = LOG.stat().st_size if LOG.exists() else 0
                if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                    return
                with LOG.open('rb') as stream:
                    stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
                missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
                if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                    action = 'remove incompatible optional torchao'
                    args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
                elif missing and missing.group(1).split('.')[0] in mapping:
                    package = mapping[missing.group(1).split('.')[0]]
                    action = 'install missing '+package
                    args = [sys.executable, '-m', 'pip', 'install', package]
                else:
                    raise RuntimeError(label+' actual import failure; see '+str(LOG))
                if action in repaired:
                    raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
                repaired.add(action)
                command(args, action)
        ensure_imports('import numpy,torch', ['numpy','torch'], 'tensor dependencies')
        ensure_imports('import accelerate,safetensors,huggingface_hub,transformers,sentencepiece,ftfy; from diffusers import WanPipeline,AutoencoderKLWan',
            ['diffusers','accelerate','safetensors','huggingface-hub','transformers','sentencepiece','ftfy'], 'Wan pipeline and VAE')
        import shutil
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            command(['apt-get','-qq','update'], 'Prepare media dependency')
            command(['apt-get','-qq','install','-y','ffmpeg'], 'Install FFmpeg')
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            raise RuntimeError('ffmpeg/ffprobe unavailable; see '+str(LOG))
        probe="from pathlib import Path; import importlib; root=Path.cwd().resolve(); names=['main.tube_state.b_adjacent_origin_confirmation_v1','runtime.wan.b_adjacent_origin_confirmation_v1','runtime.wan.b_rank3_four_source_confirmation_backend_v1','runtime.wan.b_adjacent_origin_confirmation_evaluation_v1','experiments.wan_state_clock.b_adjacent_origin_confirmation_v1_run']; modules=[importlib.import_module(n) for n in names]; assert all(Path(m.__file__).resolve().is_relative_to(root) for m in modules); print(*[m.__file__ for m in modules],sep='\\n')"
        subprocess.run([sys.executable,'-c',probe],cwd=SOURCE_ROOT,check=True)

        START_RECEIPT=reserve_start(CLOUD_CLIENT,OUTPUT)
        print('启动收据:',START_RECEIPT)
        if START_RECEIPT['status']=='START_RECEIPT_CLOUD_READ_BACK':
            args=[sys.executable,'-u','-c',"from experiments.wan_state_clock.b_adjacent_origin_confirmation_v1_run import _notebook_reserved_main; raise SystemExit(_notebook_reserved_main())",
                  '--input',str(INPUT),'--output',str(OUTPUT)]
            with (OUTPUT/'runner.log').open('a') as stream:
                worker=subprocess.Popen(args,cwd=SOURCE_ROOT,stdout=stream,stderr=subprocess.STDOUT)
                try:
                    while worker.poll() is None:
                        try:worker.wait(timeout=30)
                        except subprocess.TimeoutExpired:
                            progress=OUTPUT/'progress.json'
                            try:
                                value=json.loads(progress.read_text());print(value.get('active'),value.get('counts'),flush=True)
                            except (OSError,ValueError):print('同一worker运行中；详见runner.log。',flush=True)
                except BaseException:
                    worker.terminate()
                    try:worker.wait(timeout=10)
                    except subprocess.TimeoutExpired:worker.kill();worker.wait()
                    raise
            print('Runner exit:',worker.returncode,'保存目录:',OUTPUT)

        else:
            print('没有启动worker；云端已执行、目录未可见或收据状态未知。')
        CAN_START=False
    except BaseException as error:
        CAN_START=False
        PREPARATION_ERROR=f'{type(error).__name__}: {error}'
        print('准备或worker中断，保留原记录，仍可执行下方保存检查:',PREPARATION_ERROR)
else:
    print('跳过setup与worker：只读结果或云端状态未知。')


## 4. 系统与ADJ两层判读

系统沿原M3总体/重复/裁剪删除阈值，null检测/输出、错误和无实际支持均须为0；对RANK3R及CDC_M逐槽零丢失。系统方法达标后，相对CDC_M净增至少32才贡献达标。ADJ对RANK3R的新增/丢失和纠错分支单列；无新增则写本批未提供新增纠错证据，不自动追加来源。共同BER分母为0时未定义。


In [ ]:

if local_artifacts(OUTPUT):
    recovery=recover_ledger(OUTPUT)
    print('恢复执行状态:',recovery['counts'],'读取问题:',len(recovery['read_errors']))
    for name in ('result.json','evaluation.json'):
        try:value=json.loads((OUTPUT/name).read_text())
        except (OSError,ValueError) as error:
            print(name,'尚不可读:',type(error).__name__);continue
        if name=='result.json':print('计算状态:',value.get('status'),'固定分母:',value.get('denominators'),'物理调用:',value.get('physical_counts'),'停止:',value.get('stop'))
        else:print('系统方法:',value.get('method_status'),'贡献:',value.get('contribution_status'),'判据:',value.get('checks'),'逐源:',value.get('per_source'),'配对:',value.get('paired_summary'),'ADJ纠错证据:',value.get('adj_incremental'))
else:print('挂载端无执行记录；不能据此推断云端未开始。下方只读保存检查。')
print('完整计划/失败/质量:',OUTPUT,'四个新登记v来源，固定五种片段；历史/当前阶段分列，不能称IID或一般重复鲁棒。')


## 5. 保存与证据范围

Drive仅保留必要JSON和72个最终MP4，不新增未来任务的OFF原始大体积归档。保存状态与方法状态分列；下载发起不等于浏览器已经保存。本次固定来源仅描述性确认，不给IID、总体FPR、全攻击或感知质量PASS。代理未运行GPU。


In [ ]:

STORAGE_REPORT=confirm_and_backup(CLOUD_CLIENT,OUTPUT,backup_directory='/content',progress=lambda text:print(text,flush=True))
print(json.dumps(storage_summary(STORAGE_REPORT),ensure_ascii=False,indent=2))
print('具体缺失/读取错误列表:',STORAGE_REPORT.get('report_path'))
backup_path=STORAGE_REPORT.get('backup_zip')
if backup_path and Path(backup_path).is_file() and zipfile.is_zipfile(backup_path):
    print('本地结果备份已生成:',backup_path)
    try:
        from google.colab import files
        files.download(backup_path)
        print('已发起下载；请自行确认浏览器下载完成。')
    except Exception as error:
        print('未能发起浏览器下载，可从Colab文件面板下载该/content文件:',type(error).__name__,str(error))
else:print('未生成可打开的ZIP，具体磁盘/读取问题见保存报告。')
